# DFCI External Validation — Time-Dependent Endpoint Agreement

DFCI counterpart to `Time_Dependent_Validation_S1J_K.ipynb`. Same two analyses, **30-day horizon**.

**Panel 1 — Time-integrated Brier disagreement (30 days).** For each toxicity the LLM pipeline and the gold standard are two binary event-by-time-*t* trajectories. A unit contributes at *t* only when its status is observable in **both** arms. For binary endpoints the squared error is exactly an agreement/disagreement indicator, so `BS(t) = mean[(Y_LLM(t) - Y_GS(t))^2]` is the disagreement rate. Integrated over **30 days from `io_start`**; lower is better. Compared against a prevalence-matched random null endpoint.

**Panel 2 — Onset agreement among concordant events.** Restricted to courses both arms flagged, so the comparison isolates timing. This panel is not truncated at 30 days: concordant onsets can fall later when the irAE-positive arm's notes were sampled after first irAE onset rather than after treatment start.

## How the 100-patient note set was built

The evaluation cohort was filtered to immunotherapy patients, then split into two arms of 50:

- **irAE-positive:** gold standard had an exact case-insensitive match to one of the six target irAEs. Notes restricted to a **30-day window after that patient's first recorded irAE onset**.
- **irAE-negative:** notes restricted to a **30-day window after treatment start**, so the observation period is comparable across arms.

Patients needed at least one note in their window. Multi-row fragments were concatenated per document and merged with patient-level metadata. That is why the headline Brier is 30 days, not 12 months: 12-month follow-up was never in the input.

## How this differs from the MSKCC notebook

| | MSKCC | DFCI |
|---|---|---|
| Unit of analysis | (patient, line of therapy), n≈17,271 | (patient, ICI course), **n=100** |
| LLM output | per-batch **probabilities** + `window_start` | **categorical** AE text + `irAE Date` |
| Event definition | first batch with p ≥ per-toxicity threshold | first note naming the toxicity |
| Anchor / censoring | `lot_start`, `t_cutoff_lot` from master tables | `io_start`; notes sampled in 30-day windows |
| Brier horizon | 12 months | **30 days** |

Consequences worth keeping in view:

1. **No thresholds and no calibration.** The DFCI output is categorical, so `AE_THRESHOLDS` has no analogue. The prevalence-matched null still applies — it uses the LLM's Kaplan–Meier prevalence, not its probabilities.
2. **Note coverage is the design, not a limitation to work around.** `censor_days` is capped at last note. A gold-standard event after notes ended is out of window, not a miss.
3. **n=100 means wide intervals.** Treat per-toxicity estimates for rare toxicities as descriptive.
4. **LLM onset falls back to note date.** `irAE Date` parses for most AE rows; the remainder fall back to the flagging note's `note_date`. The fallback share is reported below.

## Matching modes

Gold-standard labels use the exact Figure 1B `TOXICITY_MAP`. Prompt text is mapped three ways (`liberal` / `conservative` / `direct`); **conservative** is the send mode. Per-mode files live under `RESULTS_DIR/{mode}/`; conservative copies are also written at `RESULTS_DIR` root.

## Outputs
Written to `../results/supp/S1J_time_dependent_validation_DFCI/`:
- `Brier_Score_DFCI.pdf` / `Brier_Score_DFCI_with_null.csv` (30-day IBS, conservative)
- `KM_Concordant_DFCI_{toxicity}.pdf` / `KM_Concordant_DFCI_all6.pdf` / `Onset_Agreement_DFCI_results.csv`
- `Horizon_Sensitivity_DFCI.csv` (30 / 90 / 180 / 365 days; only 30 days is for reporting)
- `Matching_Mode_Comparison_DFCI.csv` plus `liberal/` `conservative/` `direct/` subfolders

In [ ]:
import os
import re
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from scipy.stats import wilcoxon

%matplotlib inline

warnings.filterwarnings('ignore')

# Style block matched to Time_Dependent_Validation_S1J_K.ipynb
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
NOTEBOOK_DIR = os.getcwd()
# Data was moved out of v1/figure 1/data into figures/figures_data/figure 1/data
_data_candidates = [
    os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..', 'figures_data', 'figure 1', 'data')),
    os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'data')),
]
DATA_DIR = next((p for p in _data_candidates if os.path.isdir(p)), _data_candidates[0])
RESULTS_DIR = os.path.normpath(
    os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S1J_time_dependent_validation_DFCI'))
os.makedirs(RESULTS_DIR, exist_ok=True)

LLM_PATH      = os.path.join(DATA_DIR, 'dfci_combined.csv')
_gs_candidates = [
    os.path.join(DATA_DIR, '2026may01_merged_ae_with_apr_full.csv'),
    os.path.join(DATA_DIR, '2026May01_merged_ae_with_apr_full.csv'),
]
GS_PATH = next((p for p in _gs_candidates if os.path.exists(p)), _gs_candidates[0])
FOLLOWUP_PATH = os.path.join(DATA_DIR, 'dfci_followup.csv')   # optional

BRIER_PDF_OUT = os.path.join(RESULTS_DIR, 'Brier_Score_DFCI.pdf')
NULL_CSV_OUT  = os.path.join(RESULTS_DIR, 'Brier_Score_DFCI_with_null.csv')
KM_PDF_OUT    = os.path.join(RESULTS_DIR, 'KM_Concordant_DFCI.pdf')
KM_CSV_OUT    = os.path.join(RESULTS_DIR, 'Onset_Agreement_DFCI_results.csv')
HORIZON_CSV   = os.path.join(RESULTS_DIR, 'Horizon_Sensitivity_DFCI.csv')
COMPARE_CSV   = os.path.join(RESULTS_DIR, 'Matching_Mode_Comparison_DFCI.csv')

def mode_dir(mode):
    d = os.path.join(RESULTS_DIR, mode)
    os.makedirs(d, exist_ok=True)
    return d

print(f'DATA_DIR    : {DATA_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}\n')
for label, p, required in [('LLM output  ', LLM_PATH, True),
                           ('Gold standard', GS_PATH, True),
                           ('Follow-up   ', FOLLOWUP_PATH, False)]:
    tag = 'FOUND   ' if os.path.exists(p) else ('MISSING ' if required else 'absent  ')
    print(f'{tag} {label}  {p}')

_missing = [p for p in (LLM_PATH, GS_PATH) if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError(
        'Required input(s) not found:\n  ' + '\n  '.join(_missing) +
        '\n\nSee the table in the cell above for what each file must contain.'
    )
HAVE_FOLLOWUP = os.path.exists(FOLLOWUP_PATH)

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
TOXICITY_COLUMNS = ['pneumonitis', 'adrenal_insufficiency', 'liver_toxicity',
                    'colitis', 'hyperthyroidism', 'hypothyroidism']
TOXICITY_DISPLAY = {
    'pneumonitis': 'Pneumonitis', 'adrenal_insufficiency': 'Adrenal Insufficiency',
    'liver_toxicity': 'Liver Toxicity', 'colitis': 'Colitis',
    'hyperthyroidism': 'Hyperthyroidism', 'hypothyroidism': 'Hypothyroidism',
}

TOXICITY_MAP = {
    'pneumonitis': 'pneumonitis',
    'adrenal insufficiency': 'adrenal_insufficiency',
    'adrenal_insufficiency': 'adrenal_insufficiency',
    'liver toxicity': 'liver_toxicity',
    'liver_toxicity': 'liver_toxicity',
    'colitis': 'colitis',
    'hyperthyroidism': 'hyperthyroidism',
    'hypothyroidism': 'hypothyroidism',
}

LLM_KEYWORDS_LIBERAL = {
    'colitis': ('colitis', 'enterocolitis', 'enteritis', 'autoimmune enteritis',
                'ileitis', 'ileus', 'gastroenteritis', 'diarrhea'),
    'pneumonitis': ('pneumonitis', 'interstitial lung disease', 'ild',
                    'drug-induced lung toxicity'),
    'hyperthyroidism': ('hyperthyroid', 'thyrotoxicosis'),
    'hypothyroidism': ('hypothyroid',),
    'adrenal_insufficiency': ('adrenal insufficiency', 'adrenalitis', 'hypophysitis',
                              'hypopituitarism'),
    'liver_toxicity': ('liver toxicity', 'liver injury', 'hepatitis', 'hepatotoxicity',
                       'transaminitis', 'hyperbilirubinemia', 'drug-induced liver injury',
                       'dili', 'elevated alt', 'alt elevation', 'elevated ast',
                       'ast elevation', 'immune-mediated hepatitis',
                       'immune-related hepatitis', 'autoimmune hepatitis'),
}
LLM_KEYWORDS_CONSERVATIVE = {
    'colitis': ('colitis', 'enterocolitis', 'autoimmune enteritis'),
    'pneumonitis': ('pneumonitis',),
    'hyperthyroidism': ('hyperthyroidism', 'thyrotoxicosis'),
    'hypothyroidism': ('hypothyroidism',),
    'adrenal_insufficiency': ('adrenal insufficiency', 'adrenalitis'),
    'liver_toxicity': ('liver toxicity', 'hepatitis', 'hepatotoxicity',
                       'drug-induced liver injury', 'dili', 'immune-mediated hepatitis',
                       'immune-related hepatitis', 'autoimmune hepatitis'),
}

MATCHING_MODES  = ('liberal', 'conservative', 'direct')
PRIMARY_MODE    = 'conservative'   # copies at RESULTS_DIR root are this mode
ATTRIBUTED_ONLY = False       # True -> keep only GS rows with a non-blank Attribution
BRIER_HORIZON_DAYS = 30       # primary IBS horizon; matches how DFCI notes were sampled
KM_XLIM_MONTHS  = 12.0        # KM can show later concordant onsets (positive arm is AE-anchored)
HORIZONS_DAYS   = [30, 90, 180, 365]   # sensitivity only; report 30
CAP_AT_NOTE_COVERAGE = True   # censor at last available note
N_BOOTSTRAP, N_PERMUTE, RNG_SEED = 500, 2000, 0

_MATCHING_KEYWORDS = {
    'liberal': LLM_KEYWORDS_LIBERAL,
    'conservative': LLM_KEYWORDS_CONSERVATIVE,
    'direct': None,
}
print(f'LLM matching modes   : {", ".join(MATCHING_MODES)}')
print(f'Primary (send) mode  : {PRIMARY_MODE}')
print(f'GS attribution filter: {"attributed only" if ATTRIBUTED_ONLY else "all GS rows"}')
print(f'Censor at note cover : {CAP_AT_NOTE_COVERAGE}')
print(f'Brier horizon        : {BRIER_HORIZON_DAYS} days')


def standardize_mrn(mrn):
    """Digits only, leading zeros dropped. DFCI MRNs are 5-6 digits, so unlike the
    MSKCC notebook this does not zero-pad to 8."""
    if pd.isna(mrn):
        return None
    digits = re.findall(r'\d+', str(mrn).strip().strip("'\""))
    if not digits:
        return None
    try:
        return str(int(''.join(digits)))
    except (ValueError, TypeError):
        return None


PLACEHOLDERS = {'not applicable', 'not sure', 'none', 'nan', ''}


def map_llm_events(value, mode):
    """Map a DFCI free-text Adverse Event cell to zero or more of the six irAEs.

    liberal / conservative: substring keywords from Dana_Farber_Metrics.py.
    direct: same exact labels as Figure 1B. Compound cells are split on
    commas / slashes / 'and' so 'colitis and hypothyroidism' can hit both;
    synonyms (diarrhea, transaminitis, hepatitis, …) are not expanded.
    """
    if pd.isna(value):
        return []
    text = str(value).strip().lower()
    if text in PLACEHOLDERS:
        return []
    if mode == 'direct':
        hits = []
        mapped = TOXICITY_MAP.get(text)
        if mapped:
            hits.append(mapped)
        for part in re.split(r'[,;/]|\band\b', text):
            mapped = TOXICITY_MAP.get(part.strip())
            if mapped and mapped not in hits:
                hits.append(mapped)
        return hits
    keywords = _MATCHING_KEYWORDS[mode]
    return [c for c, kws in keywords.items() if any(kw in text for kw in kws)]

## Backbone — one row per patient-course, with censoring

The MSKCC notebook gets its scaffolding from the per-toxicity master tables (`lot_start`,
`t_cutoff_lot`). DFCI has no line-of-therapy structure and no pre-computed cutoff, so the backbone
is built from `combined.csv`: `io_start` is the time anchor, and every patient has exactly one, so
the unit of analysis is the patient.

`censor_days` resolves in this order:
1. `censor_days` column in `dfci_followup.csv`, if present
2. `last_followup_date` in `dfci_followup.csv`, minus `io_start`
3. last available `note_date` minus `io_start` — **fallback only**

With `CAP_AT_NOTE_COVERAGE = True` the result is additionally capped at note coverage regardless of
source. This is deliberate: the pipeline read only these notes, so a gold-standard event after the
last note is not a detection failure the model could have avoided. Leaving the cap off inflates
apparent disagreement in the late window. Both are reported.

In [ ]:
llm = pd.read_csv(LLM_PATH, low_memory=False)
llm.columns = [str(c).strip() for c in llm.columns]
llm['mrn'] = llm['mrn'].apply(standardize_mrn)
llm = llm[llm['mrn'].notna()].copy()

for col in ('note_date', 'io_start'):
    if col not in llm.columns:
        raise KeyError(f'LLM file is missing required column `{col}`')
    llm[col] = pd.to_datetime(llm[col], errors='coerce')

if 'irAE Date' in llm.columns:
    llm['irae_date'] = pd.to_datetime(llm['irAE Date'], errors='coerce')
else:
    llm['irae_date'] = pd.NaT
    print('WARNING: no `irAE Date` column — LLM onset will use note_date for every row')

n_patients = llm['mrn'].nunique()
print(f'LLM rows: {len(llm):,}  patients: {n_patients:,}  notes: {llm["doc"].nunique():,}')

# One ICI course per patient
course = (llm.groupby('mrn', as_index=False)
             .agg(io_start=('io_start', 'min'),
                  note_min=('note_date', 'min'),
                  note_max=('note_date', 'max'),
                  n_notes=('doc', 'nunique'),
                  n_rows=('mrn', 'size')))
course['note_coverage_days'] = (course['note_max'] - course['io_start']).dt.days
print('\nNote coverage (days past io_start):')
print(course['note_coverage_days'].describe().to_string())

# Optional real follow-up
if HAVE_FOLLOWUP:
    fu = pd.read_csv(FOLLOWUP_PATH, low_memory=False)
    fu.columns = [str(c).strip() for c in fu.columns]
    mrn_col = next((c for c in fu.columns if c.lower() == 'mrn'), None)
    if mrn_col is None:
        raise KeyError('dfci_followup.csv needs an MRN column')
    fu['mrn'] = fu[mrn_col].apply(standardize_mrn)
    fu = fu[fu['mrn'].notna()].drop_duplicates('mrn')

    if 'censor_days' in fu.columns:
        fu['censor_days'] = pd.to_numeric(fu['censor_days'], errors='coerce')
        course = course.merge(fu[['mrn', 'censor_days']], on='mrn', how='left')
        source = 'censor_days column'
    else:
        date_col = next((c for c in fu.columns
                         if c.lower() in ('last_followup_date', 'last_fu', 'dod', 'censor_date')), None)
        if date_col is None:
            raise KeyError('dfci_followup.csv needs `censor_days` or `last_followup_date`')
        fu['last_fu'] = pd.to_datetime(fu[date_col], errors='coerce')
        course = course.merge(fu[['mrn', 'last_fu']], on='mrn', how='left')
        course['censor_days'] = (course['last_fu'] - course['io_start']).dt.days
        source = date_col
    print(f'\nFollow-up loaded from {source} ({course["censor_days"].notna().sum()} / {len(course)} patients)')
else:
    course['censor_days'] = course['note_coverage_days']
    print('\nWARNING: dfci_followup.csv not found. Censoring = last note − io_start. '
          'That is a lower bound on true follow-up. Supply the real file before reporting.')

course['censor_days_uncapped'] = course['censor_days']
if CAP_AT_NOTE_COVERAGE:
    course['censor_days'] = np.minimum(course['censor_days'], course['note_coverage_days'])
    n_capped = (course['censor_days'] < course['censor_days_uncapped']).sum()
    print(f'Capped {n_capped} patients at note coverage (CAP_AT_NOTE_COVERAGE=True)')

backbone = course.loc[
    np.isfinite(course['censor_days']) & (course['censor_days'] > 0),
    ['mrn', 'io_start', 'censor_days', 'censor_days_uncapped', 'note_coverage_days', 'n_notes']
].copy()
print(f'\nBackbone: {len(backbone):,} patients with valid positive censoring')
print(f'  median censor_days          = {backbone["censor_days"].median():.0f}')
print(f'  median note_coverage_days   = {backbone["note_coverage_days"].median():.0f}')
print(f'  median uncapped follow-up   = {backbone["censor_days_uncapped"].median():.0f}')

## Gold standard

Uses the same `2026may01_merged_ae_with_apr_full.csv` as S1J. That file is the institutional chart-review extract (176k rows, 8,116 patients). All 100 DFCI MRNs are in it. The backbone is built from `dfci_combined.csv` first, so the extra ~8,000 patients never enter the analysis.

`Toxicity` is matched **exactly** as in Figure 1B (`TOXICITY_MAP`): `pneumonitis`, `adrenal insufficiency`, `liver toxicity`, `colitis`, `hyperthyroidism`, `hypothyroidism`. Synonyms are not expanded. Liberal / conservative / direct matching applies only to DFCI prompt text.

Required columns already present: `MRN`, `Toxicity`, `Start Date`. `Attribution` is used only if `ATTRIBUTED_ONLY=True`. `APR_LOT` is ignored here — DFCI has no line-of-therapy structure, so events are matched on MRN + days from `io_start`.

In [ ]:
gs_raw = pd.read_csv(GS_PATH, encoding='latin-1', low_memory=False)
gs_raw.columns = [str(c).strip() for c in gs_raw.columns]

def _col(df, *names):
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None

mrn_col  = _col(gs_raw, 'MRN', 'mrn')
tox_col  = _col(gs_raw, 'Toxicity', 'toxicity', 'Adverse Event', 'ae')
date_col = _col(gs_raw, 'Start Date', 'Start_Date', 'ae_date', 'AE Date', 'onset_date')
attr_col = _col(gs_raw, 'Attribution', 'attribution')
if mrn_col is None or tox_col is None or date_col is None:
    raise KeyError(
        'Gold standard needs MRN, Toxicity, and Start Date. '
        f'Found columns: {list(gs_raw.columns)}'
    )

gs = gs_raw.copy()
gs['mrn'] = gs[mrn_col].apply(standardize_mrn)
gs = gs[gs['mrn'].notna()].copy()
gs['Start Date'] = pd.to_datetime(gs[date_col], errors='coerce')
gs = gs[gs['Start Date'].notna()].copy()

if ATTRIBUTED_ONLY:
    if attr_col is None:
        raise KeyError('ATTRIBUTED_ONLY=True but no Attribution column in the gold standard')
    n_before = len(gs)
    gs = gs[gs[attr_col].notna() & (gs[attr_col].astype(str).str.strip() != '')].copy()
    print(f'Attribution filter: {len(gs):,} / {n_before:,} rows retained')

gs['tox_lower'] = gs[tox_col].fillna('').astype(str).str.lower().str.strip()
gs['tox_mapped'] = gs['tox_lower'].map(TOXICITY_MAP)
gs_mapped = gs[gs['tox_mapped'].notna()].copy()
print(f'Gold standard: {len(gs):,} dated rows, {gs["mrn"].nunique():,} patients')
print(f'Exact Fig 1B labels among 6 irAEs: {len(gs_mapped):,} rows, {gs_mapped["mrn"].nunique():,} patients')
print(gs_mapped['tox_mapped'].value_counts().rename(index=TOXICITY_DISPLAY).to_string())

# Reviewed-cohort restriction — same logic as the MSKCC notebook
gs_reviewed_mrns = set(gs['mrn'].dropna())
n_before = len(backbone)
backbone = backbone[backbone['mrn'].isin(gs_reviewed_mrns)].copy()
print(f'\nReviewed-cohort filter: {len(backbone):,} / {n_before:,} backbone patients remain '
      f'({backbone["mrn"].nunique():,} unique)')
missing_llm = sorted(set(llm['mrn']) - gs_reviewed_mrns)
if missing_llm:
    print(f'WARNING: {len(missing_llm)} LLM patients have no gold-standard record and were dropped: '
          f'{missing_llm[:10]}{"..." if len(missing_llm) > 10 else ""}')

# Attach course dates and keep events inside the course window
gs_mapped = gs_mapped.merge(backbone[['mrn', 'io_start', 'censor_days']], on='mrn', how='inner')
gs_mapped['days_from_io'] = (gs_mapped['Start Date'] - gs_mapped['io_start']).dt.days
n_neg = (gs_mapped['days_from_io'] < 0).sum()
n_late = (gs_mapped['days_from_io'] > gs_mapped['censor_days']).sum()
print(f'Excluding {n_neg:,} GS events before io_start and {n_late:,} after censor_days')
gs_mapped = gs_mapped[(gs_mapped['days_from_io'] >= 0) &
                      (gs_mapped['days_from_io'] <= gs_mapped['censor_days'])].copy()
print(f'{len(gs_mapped):,} GS AE records inside the course window')

## Build GS and LLM (time, event) per toxicity

GS onset = earliest `Start Date` after `io_start` whose `Toxicity` is an exact Figure 1B label. That mapping does not depend on matching mode.

LLM onset = earliest mapped AE. `irAE Date` is used when it parses; otherwise `note_date`. All three matching modes are built here (`liberal` / `conservative` / `direct`). A single `Adverse Event` cell can name more than one toxicity, so every hit is kept.

In [ ]:
def build_gs_survival(tox):
    rec = gs_mapped[gs_mapped['tox_mapped'] == tox]
    first = (rec.groupby('mrn')['days_from_io'].min()
                .reset_index().rename(columns={'days_from_io': 'gs_onset_days'}))
    surv = backbone[['mrn', 'censor_days']].merge(first, on='mrn', how='left')
    surv['event'] = surv['gs_onset_days'].notna().astype(int)
    surv['time'] = np.where(surv['event'] == 1, surv['gs_onset_days'], surv['censor_days'])
    return surv[surv['time'] > 0].reset_index(drop=True)


gs_survival = {tox: build_gs_survival(tox) for tox in TOXICITY_COLUMNS}


def build_llm_for_mode(mode):
    llm_exp = llm.merge(backbone[['mrn', 'censor_days']], on='mrn', how='inner')
    llm_exp['std_events'] = llm_exp['Adverse Event'].apply(lambda v: map_llm_events(v, mode))
    llm_exp = llm_exp.explode('std_events')
    llm_exp = llm_exp[llm_exp['std_events'].isin(TOXICITY_COLUMNS)].copy()
    llm_exp['onset_date'] = llm_exp['irae_date'].fillna(llm_exp['note_date'])
    n_fallback = llm_exp['irae_date'].isna().sum()
    print(f'\n=== LLM matching: {mode} ===')
    print(f'  mapped AE rows after explode: {len(llm_exp):,}')
    print(f'  using irAE Date : {llm_exp["irae_date"].notna().sum():,}')
    print(f'  fallback to note_date: {n_fallback:,} ({100 * n_fallback / max(len(llm_exp), 1):.1f}%)')
    print(llm_exp['std_events'].value_counts().rename(index=TOXICITY_DISPLAY).to_string())
    llm_exp['days_from_io'] = (llm_exp['onset_date'] - llm_exp['io_start']).dt.days
    n_neg = (llm_exp['days_from_io'] < 0).sum()
    n_late = (llm_exp['days_from_io'] > llm_exp['censor_days']).sum()
    print(f'  excluding {n_neg:,} before io_start and {n_late:,} after censor_days')
    llm_in = llm_exp[(llm_exp['days_from_io'] >= 0) &
                     (llm_exp['days_from_io'] <= llm_exp['censor_days'])].copy()

    def _build(tox):
        rec = llm_in[llm_in['std_events'] == tox]
        first = (rec.groupby('mrn')['days_from_io'].min()
                    .reset_index().rename(columns={'days_from_io': 'llm_onset_days'}))
        surv = backbone[['mrn', 'censor_days']].merge(first, on='mrn', how='left')
        surv['event'] = surv['llm_onset_days'].notna().astype(int)
        surv['time'] = np.where(surv['event'] == 1, surv['llm_onset_days'], surv['censor_days'])
        return surv[surv['time'] > 0].reset_index(drop=True)

    return {tox: _build(tox) for tox in TOXICITY_COLUMNS}, llm_in


llm_survival_by_mode = {}
llm_in_window_by_mode = {}
for _mode in MATCHING_MODES:
    llm_survival_by_mode[_mode], llm_in_window_by_mode[_mode] = build_llm_for_mode(_mode)

print(f'\n{"Toxicity":<24} {"GS":>4} {"GS≤30d":>7} | {"lib":>5} {"con":>5} {"dir":>5} | {"lib30":>5} {"con30":>5} {"dir30":>5}')
print('-' * 80)
for tox in TOXICITY_COLUMNS:
    g = gs_survival[tox]
    gs_all = int(g['event'].sum())
    gs30 = int(((g['event'] == 1) & (g['gs_onset_days'] <= BRIER_HORIZON_DAYS)).sum())
    counts = {}
    for mode in MATCHING_MODES:
        l = llm_survival_by_mode[mode][tox]
        counts[mode] = (
            int(l['event'].sum()),
            int(((l['event'] == 1) & (l['llm_onset_days'] <= BRIER_HORIZON_DAYS)).sum()),
        )
    print(f'{TOXICITY_DISPLAY[tox]:<24} {gs_all:>4} {gs30:>7} | '
          f'{counts["liberal"][0]:>5} {counts["conservative"][0]:>5} {counts["direct"][0]:>5} | '
          f'{counts["liberal"][1]:>5} {counts["conservative"][1]:>5} {counts["direct"][1]:>5}')

## Time-integrated Brier disagreement + prevalence-matched null

30-day IBS for **all three** matching modes. The unit is the patient. Bootstrap CIs and 2,000 permutations are computed per mode. Copies under `RESULTS_DIR/{mode}/`; the conservative copies are also written at `RESULTS_DIR` root for sending.

Do not report 90/180/365-day IBS, RelImpr when Null≈0, or hyperthyroidism/pneumonitis 30-day Brier as “perfect performance” (those events fall after day 30).

In [ ]:
def _integrated_brier_endpoint_agreement(gs_times, gs_events, gs_censor,
                                           llm_times, llm_events, llm_censor,
                                           time_grid):
    gs_times = np.asarray(gs_times, float)
    gs_events = np.asarray(gs_events, int)
    gs_censor = np.asarray(gs_censor, float)
    llm_times = np.asarray(llm_times, float)
    llm_events = np.asarray(llm_events, int)
    llm_censor = np.asarray(llm_censor, float)

    valid = (np.isfinite(gs_times) & np.isfinite(gs_censor) &
             np.isfinite(llm_times) & np.isfinite(llm_censor))
    if not valid.any():
        return np.nan, 0, np.array([]), np.array([])

    gs_times, gs_events, gs_censor = gs_times[valid], gs_events[valid], gs_censor[valid]
    llm_times, llm_events, llm_censor = llm_times[valid], llm_events[valid], llm_censor[valid]

    bs, vt, n_obs = [], [], []
    for t in time_grid:
        gs_obs = ((gs_events == 1) & (gs_times <= t)) | (gs_censor > t)
        llm_obs = ((llm_events == 1) & (llm_times <= t)) | (llm_censor > t)
        mask = gs_obs & llm_obs
        if not mask.any():
            continue
        y_gs = ((gs_events == 1) & (gs_times <= t))[mask].astype(int)
        y_llm = ((llm_events == 1) & (llm_times <= t))[mask].astype(int)
        bs.append(np.mean((y_llm - y_gs) ** 2))
        vt.append(t)
        n_obs.append(int(mask.sum()))

    if len(bs) < 2:
        return np.nan, len(bs), np.asarray(vt), np.asarray(n_obs)
    _trapz = getattr(np, 'trapezoid', None) or np.trapz
    rng_t = vt[-1] - vt[0]
    if rng_t <= 0:
        return np.nan, len(vt), np.asarray(vt), np.asarray(n_obs)
    ibs = _trapz(bs, vt) / rng_t
    return float(max(0.0, min(1.0, ibs))), len(vt), np.asarray(vt), np.asarray(n_obs)


def _integrated_brier_null_prevalence_matched(gs_times, gs_events, gs_censor,
                                               llm_times, llm_events, llm_censor,
                                               time_grid):
    gs_times = np.asarray(gs_times, float)
    gs_events = np.asarray(gs_events, int)
    gs_censor = np.asarray(gs_censor, float)
    llm_times = np.asarray(llm_times, float)
    llm_events = np.asarray(llm_events, int)
    llm_censor = np.asarray(llm_censor, float)

    valid = (np.isfinite(gs_times) & np.isfinite(gs_censor) &
             np.isfinite(llm_times) & np.isfinite(llm_censor))
    if not valid.any():
        return np.nan
    gs_times, gs_events, gs_censor = gs_times[valid], gs_events[valid], gs_censor[valid]
    llm_times, llm_events, llm_censor = llm_times[valid], llm_events[valid], llm_censor[valid]

    kmf_llm = KaplanMeierFitter()
    kmf_llm.fit(llm_times, llm_events)

    bs, vt = [], []
    for t in time_grid:
        gs_obs = ((gs_events == 1) & (gs_times <= t)) | (gs_censor > t)
        llm_obs = ((llm_events == 1) & (llm_times <= t)) | (llm_censor > t)
        mask = gs_obs & llm_obs
        if not mask.any():
            continue
        y_gs = ((gs_events == 1) & (gs_times <= t))[mask].astype(int)
        n = len(y_gs)
        p_llm = 1.0 - float(kmf_llm.predict(t))
        expected_bs = (y_gs.sum() * (1 - p_llm) + (n - y_gs.sum()) * p_llm) / n
        bs.append(expected_bs)
        vt.append(t)
    if len(bs) < 2:
        return np.nan
    _trapz = getattr(np, 'trapezoid', None) or np.trapz
    rng_t = vt[-1] - vt[0]
    if rng_t <= 0:
        return np.nan
    return float(max(0.0, min(1.0, _trapz(bs, vt) / rng_t)))


def _align(tox, mode):
    g = gs_survival[tox].set_index('mrn')
    l = llm_survival_by_mode[mode][tox].set_index('mrn')
    return g[['gs_onset_days', 'event', 'time', 'censor_days']].join(
        l[['llm_onset_days', 'event', 'time', 'censor_days']].rename(columns={
            'event': 'llm_event', 'time': 'llm_time', 'censor_days': 'llm_censor_days'
        }),
        how='inner'
    )


def _run_ibs(aligned, time_grid, n_boot=N_BOOTSTRAP, n_perm=N_PERMUTE, seed=RNG_SEED):
    point, n_grid, used_t, n_obs = _integrated_brier_endpoint_agreement(
        aligned['time'].values, aligned['event'].values, aligned['censor_days'].values,
        aligned['llm_time'].values, aligned['llm_event'].values, aligned['llm_censor_days'].values,
        time_grid)
    null_ibs = _integrated_brier_null_prevalence_matched(
        aligned['time'].values, aligned['event'].values, aligned['censor_days'].values,
        aligned['llm_time'].values, aligned['llm_event'].values, aligned['llm_censor_days'].values,
        time_grid)

    groups = aligned.index.values
    uniq = np.unique(groups)
    idx_by_pt = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(seed)
    boot_llm = np.empty(n_boot)
    boot_null = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_pt[g] for g in pick])
        boot_llm[b] = _integrated_brier_endpoint_agreement(
            aligned['time'].values[idx], aligned['event'].values[idx], aligned['censor_days'].values[idx],
            aligned['llm_time'].values[idx], aligned['llm_event'].values[idx],
            aligned['llm_censor_days'].values[idx], time_grid)[0]
        boot_null[b] = _integrated_brier_null_prevalence_matched(
            aligned['time'].values[idx], aligned['event'].values[idx], aligned['censor_days'].values[idx],
            aligned['llm_time'].values[idx], aligned['llm_event'].values[idx],
            aligned['llm_censor_days'].values[idx], time_grid)

    perm_ibs = np.empty(n_perm)
    for p in range(n_perm):
        perm_idx = rng.permutation(len(aligned))
        perm_ibs[p] = _integrated_brier_endpoint_agreement(
            aligned['time'].values, aligned['event'].values, aligned['censor_days'].values,
            aligned['llm_time'].values[perm_idx], aligned['llm_event'].values[perm_idx],
            aligned['llm_censor_days'].values[perm_idx], time_grid)[0]
    p_value = (np.sum(perm_ibs <= point) + 1) / (n_perm + 1)

    ci_lo, ci_hi = np.nanpercentile(boot_llm, [2.5, 97.5])
    null_lo, null_hi = np.nanpercentile(boot_null, [2.5, 97.5])
    boot_diff = boot_llm - boot_null
    boot_rel = (boot_null - boot_llm) / boot_null
    return {
        'time_integrated_brier': point,
        'ci_95_lower': ci_lo, 'ci_95_upper': ci_hi,
        'null_ibs': null_ibs, 'null_ibs_ci_lower': null_lo, 'null_ibs_ci_upper': null_hi,
        'ibs_difference': point - null_ibs if np.isfinite(point) and np.isfinite(null_ibs) else np.nan,
        'ibs_diff_ci_lower': np.nanpercentile(boot_diff, 2.5),
        'ibs_diff_ci_upper': np.nanpercentile(boot_diff, 97.5),
        'relative_improvement': ((null_ibs - point) / null_ibs
                                 if np.isfinite(null_ibs) and null_ibs > 0 else np.nan),
        'rel_improvement_ci_lower': np.nanpercentile(boot_rel, 2.5),
        'rel_improvement_ci_upper': np.nanpercentile(boot_rel, 97.5),
        'p_value_vs_null': p_value,
        'n_grid_points_used': n_grid,
        'n_patients': len(uniq),
        'n_gs_events': int(aligned['event'].sum()),
        'n_llm_events': int(aligned['llm_event'].sum()),
        'mean_joint_observable': float(np.mean(n_obs)) if len(n_obs) else np.nan,
    }


TIME_GRID_DAYS = np.linspace(0, BRIER_HORIZON_DAYS, 50)

print('Joint-observability audit (pneumonitis, primary mode):')
_a = _align('pneumonitis', PRIMARY_MODE)
for _t in HORIZONS_DAYS:
    _gs_obs = _a['gs_onset_days'].notna() & (_a['gs_onset_days'] <= _t) | (_a['censor_days'] > _t)
    _llm_obs = _a['llm_onset_days'].notna() & (_a['llm_onset_days'] <= _t) | (_a['llm_censor_days'] > _t)
    _mask = _gs_obs & _llm_obs
    print(f'  t={_t:3d}d  joint-observable={int(_mask.sum()):3d}  excluded={len(_a) - int(_mask.sum()):3d}')

all_brier = []
brier_by_mode = {}
for mode in MATCHING_MODES:
    print(f'\nIBS vs prevalence-matched null  (horizon={BRIER_HORIZON_DAYS:.0f} d, matching={mode})')
    records = []
    for tox in TOXICITY_COLUMNS:
        aligned = _align(tox, mode)
        rec = _run_ibs(aligned, TIME_GRID_DAYS)
        rec.update({'toxicity': tox, 'toxicity_display': TOXICITY_DISPLAY[tox],
                    'horizon_days': BRIER_HORIZON_DAYS, 'matching': mode})
        records.append(rec)
        rel = rec['relative_improvement']
        rel_s = f'{100 * rel:+.1f}%' if np.isfinite(rel) else 'nan%'
        print(f'  {TOXICITY_DISPLAY[tox]:22s} LLM={rec["time_integrated_brier"]:.4f}  '
              f'Null={rec["null_ibs"]:.4f}  Diff={rec["ibs_difference"]:+.4f}  '
              f'RelImpr={rel_s}  p={rec["p_value_vs_null"]:.4f}  '
              f'n={rec["n_patients"]}  GSev={rec["n_gs_events"]}  LLMev={rec["n_llm_events"]}')
    brier_df = pd.DataFrame(records)
    brier_by_mode[mode] = brier_df
    out = os.path.join(mode_dir(mode), 'Brier_Score_DFCI_with_null.csv')
    brier_df.to_csv(out, index=False)
    print(f'Saved: {out}')
    all_brier.append(brier_df)

brier_all = pd.concat(all_brier, ignore_index=True)
brier_all.to_csv(os.path.join(RESULTS_DIR, 'Brier_Score_DFCI_all_modes.csv'), index=False)
brier_df = brier_by_mode[PRIMARY_MODE]
brier_df.to_csv(NULL_CSV_OUT, index=False)
print(f'\nPrimary ({PRIMARY_MODE}) also saved as {os.path.basename(NULL_CSV_OUT)}')
brier_df[['toxicity_display', 'time_integrated_brier', 'null_ibs',
          'ibs_difference', 'relative_improvement', 'p_value_vs_null',
          'n_gs_events', 'n_llm_events']]

## Horizon sensitivity

The reported number is **30-day** IBS. Longer horizons are printed only to show that 90 / 180 / 365 days are not the DFCI design (notes were sampled in 30-day windows). Do not report them as the result.

In [ ]:
horizon_all = []
for mode in MATCHING_MODES:
    horizon_rows = []
    for h in HORIZONS_DAYS:
        grid = np.linspace(0, h, 50)
        for tox in TOXICITY_COLUMNS:
            aligned = _align(tox, mode)
            point, n_grid, _, n_obs = _integrated_brier_endpoint_agreement(
                aligned['time'].values, aligned['event'].values, aligned['censor_days'].values,
                aligned['llm_time'].values, aligned['llm_event'].values,
                aligned['llm_censor_days'].values, grid)
            horizon_rows.append({
                'matching': mode,
                'horizon_days': h, 'toxicity': tox,
                'toxicity_display': TOXICITY_DISPLAY[tox],
                'ibs': point, 'n_grid_points_used': n_grid,
                'mean_joint_observable': float(np.mean(n_obs)) if len(n_obs) else np.nan,
                'n_patients': len(aligned),
            })
    hdf = pd.DataFrame(horizon_rows)
    hdf.to_csv(os.path.join(mode_dir(mode), 'Horizon_Sensitivity_DFCI.csv'), index=False)
    print(f'\nHorizon IBS ({mode})')
    print(hdf.pivot(index='toxicity_display', columns='horizon_days', values='ibs')
          .round(4).to_string())
    horizon_all.append(hdf)

horizon_df = pd.concat(horizon_all, ignore_index=True)
horizon_df.to_csv(os.path.join(RESULTS_DIR, 'Horizon_Sensitivity_DFCI_all_modes.csv'), index=False)
horizon_df[horizon_df['matching'] == PRIMARY_MODE].drop(columns='matching').to_csv(
    HORIZON_CSV, index=False)
print(f'\nSaved: {os.path.basename(HORIZON_CSV)} (primary={PRIMARY_MODE})')
print('Report the 30-day column only.')

## Panel 1 — Brier bar chart (30-day IBS)

In [ ]:
PANEL_TOX_ORDER = ['liver_toxicity', 'hypothyroidism', 'pneumonitis',
                   'colitis', 'adrenal_insufficiency', 'hyperthyroidism']
assert set(PANEL_TOX_ORDER) == set(TOXICITY_COLUMNS)


def plot_brier_bars(plot_src, out_path):
    plot_df = plot_src.set_index('toxicity').loc[PANEL_TOX_ORDER].reset_index()
    x = np.arange(len(plot_df))
    fig, ax = plt.subplots(figsize=(2.5112, 1.8949))
    yerr = np.array([
        plot_df['time_integrated_brier'] - plot_df['ci_95_lower'],
        plot_df['ci_95_upper'] - plot_df['time_integrated_brier'],
    ])
    ax.bar(x, plot_df['time_integrated_brier'], width=0.6, color='#4C72B0',
           edgecolor='white', linewidth=0.5, yerr=yerr, capsize=3,
           error_kw={'linewidth': 0.8})
    for i, v in enumerate(plot_df['time_integrated_brier']):
        ax.text(i, plot_df['ci_95_upper'].iloc[i] + 0.0015, f'{v:.3f}',
                ha='center', va='bottom', fontsize=5)
    ax.set_ylabel('Time-integrated\nBrier disagreement (30d)')
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['toxicity_display'], rotation=30, ha='right')
    ax.set_ylim(0, max(plot_df['ci_95_upper'].max() * 1.25, 0.05))
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('bottom', 'left'):
        ax.spines[spine].set_linewidth(0.6)
    ax.tick_params(width=0.6, length=3)
    fig.subplots_adjust(left=0.28, right=0.97, top=0.95, bottom=0.32)
    fig.savefig(out_path, dpi=450)
    plt.close(fig)
    print(f'Saved: {out_path}')


for mode in MATCHING_MODES:
    plot_brier_bars(brier_by_mode[mode],
                    os.path.join(mode_dir(mode), 'Brier_Score_DFCI.pdf'))
plot_brier_bars(brier_by_mode[PRIMARY_MODE], BRIER_PDF_OUT)

## Onset agreement among gold-standard events

Conditioned on patients the GS flagged. `medΔd` is median days between GS and LLM onset among patients both flagged; negative means the pipeline flagged earlier. Computed for all three matching modes; the send table is **conservative**.

In [ ]:
def build_detection_frame(tox, mode):
    g = gs_survival[tox]
    g = g[g['event'] == 1][['mrn', 'gs_onset_days', 'censor_days']].copy()
    l = llm_survival_by_mode[mode][tox][['mrn', 'llm_onset_days']]
    d = g.merge(l, on='mrn', how='left')
    d['llm_detected'] = d['llm_onset_days'].notna().astype(int)
    d['onset_diff'] = d['llm_onset_days'] - d['gs_onset_days']
    return d


detection_by_mode = {}
det_all = []
for mode in MATCHING_MODES:
    print(f'\nOnset agreement among GS events  (matching={mode})')
    print(f'{"Toxicity":<24} {"GSev":>6} {"detected":>9} {"sens":>6} '
          f'{"medΔd":>7} {"±30d":>6} {"±60d":>6}')
    print('-' * 70)
    detection = {}
    det_rows = []
    for tox in TOXICITY_COLUMNS:
        d = build_detection_frame(tox, mode)
        detection[tox] = d
        both = d[d['llm_detected'] == 1]
        sens = d['llm_detected'].mean() if len(d) else np.nan
        med = both['onset_diff'].median() if len(both) else np.nan
        w30 = (both['onset_diff'].abs() <= 30).mean() if len(both) else np.nan
        w60 = (both['onset_diff'].abs() <= 60).mean() if len(both) else np.nan
        print(f'{TOXICITY_DISPLAY[tox]:<24} {len(d):>6,} {len(both):>9,} '
              f'{100 * sens if np.isfinite(sens) else float("nan"):>5.1f}% '
              f'{med if np.isfinite(med) else float("nan"):>7.0f} '
              f'{100 * w30 if np.isfinite(w30) else float("nan"):>5.1f}% '
              f'{100 * w60 if np.isfinite(w60) else float("nan"):>5.1f}%')
        det_rows.append({
            'toxicity': tox, 'toxicity_display': TOXICITY_DISPLAY[tox],
            'n_gs_events': len(d), 'n_detected': len(both), 'sensitivity': sens,
            'median_onset_diff_days': med, 'pct_within_30d': w30, 'pct_within_60d': w60,
            'matching': mode,
        })
    detection_by_mode[mode] = detection
    det_df = pd.DataFrame(det_rows)
    det_df.to_csv(os.path.join(mode_dir(mode), 'Onset_Agreement_DFCI_results.csv'), index=False)
    det_all.append(det_df)

detection = detection_by_mode[PRIMARY_MODE]
detection_df = pd.concat(det_all, ignore_index=True)
detection_df.to_csv(os.path.join(RESULTS_DIR, 'Onset_Agreement_DFCI_all_modes.csv'), index=False)
detection_df[detection_df['matching'] == PRIMARY_MODE].to_csv(KM_CSV_OUT, index=False)
print(f'\nSaved: {os.path.basename(KM_CSV_OUT)} (primary={PRIMARY_MODE})')
detection_df[detection_df['matching'] == PRIMARY_MODE]

## Panel 2 — onset timing, concordant events only (all six toxicities)

Restricted to gold-standard events the prompt also flagged. Colitis has no concordant pairs under conservative matching (0/2) and is plotted as an empty panel. PDFs are written per matching mode; conservative copies also go to `RESULTS_DIR` root.

In [ ]:
def _plot_km_ax(ax, tox, detection_map):
    d_all = detection_map[tox]
    d_both = d_all[d_all['llm_detected'] == 1].copy()
    n_both, n_gs = len(d_both), len(d_all)
    ax.set_xlim(0, KM_XLIM_MONTHS)
    ax.set_ylim(0, 105)
    ax.set_title(f'{TOXICITY_DISPLAY[tox]} (DFCI)', fontsize=6, pad=2)
    if n_both < 1:
        ax.text(0.5, 0.5, f'No concordant events\n({n_both}/{n_gs} GS)',
                ha='center', va='center', fontsize=6, transform=ax.transAxes)
    else:
        _ev = np.ones(n_both)
        kmf_gs = KaplanMeierFitter().fit(d_both['gs_onset_days'] / 30.44, _ev, label='Gold Standard')
        kmf_llm = KaplanMeierFitter().fit(d_both['llm_onset_days'] / 30.44, _ev, label='LLM Pipeline')
        for kmf, color, ls in [(kmf_gs, '#1f77b4', '-'), (kmf_llm, '#ff7f0e', '--')]:
            sf = kmf.survival_function_
            ax.step(sf.index, (1 - sf.iloc[:, 0]) * 100, where='post',
                    color=color, linewidth=1.0, linestyle=ls, label=kmf._label)
        ax.legend(loc='upper left', fontsize=5, frameon=False, handlelength=1.4,
                  handletextpad=0.4, labelspacing=0.3, borderaxespad=0.2)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('bottom', 'left'):
        ax.spines[spine].set_linewidth(0.6)
    ax.tick_params(width=0.6, length=3)
    return n_both, n_gs


def write_km_set(detection_map, out_dir):
    for tox in TOXICITY_COLUMNS:
        fig, ax = plt.subplots(figsize=(2.5112, 1.8949))
        n_both, n_gs = _plot_km_ax(ax, tox, detection_map)
        ax.set_ylabel('Cumulative onset (%)')
        ax.set_xlabel('Months from ICI start')
        fig.subplots_adjust(left=0.28, right=0.97, top=0.90, bottom=0.32)
        stem = f'KM_Concordant_DFCI_{tox}'
        fig.savefig(os.path.join(out_dir, f'{stem}.pdf'), dpi=450)
        plt.close(fig)
        print(f'  {stem}.pdf  concordant={n_both}/{n_gs}')
    fig, axes = plt.subplots(2, 3, figsize=(7.4, 4.4), sharey=True)
    for ax, tox in zip(axes.ravel(), TOXICITY_COLUMNS):
        _plot_km_ax(ax, tox, detection_map)
        ax.set_xlabel('Months from ICI start', fontsize=6)
    axes[0, 0].set_ylabel('Cumulative onset (%)')
    axes[1, 0].set_ylabel('Cumulative onset (%)')
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, 'KM_Concordant_DFCI_all6.pdf'), dpi=450)
    plt.close(fig)
    print(f'  KM_Concordant_DFCI_all6.pdf')


for mode in MATCHING_MODES:
    print(f'\nKM curves ({mode})')
    write_km_set(detection_by_mode[mode], mode_dir(mode))
print(f'\nKM curves (primary copy → RESULTS_DIR, {PRIMARY_MODE})')
write_km_set(detection_by_mode[PRIMARY_MODE], RESULTS_DIR)

# Combined comparison table
cmp_rows = []
for mode in MATCHING_MODES:
    b = brier_by_mode[mode].set_index('toxicity')
    d = detection_df[detection_df['matching'] == mode].set_index('toxicity')
    for tox in TOXICITY_COLUMNS:
        g = gs_survival[tox]
        l = llm_survival_by_mode[mode][tox]
        cmp_rows.append({
            'matching': mode,
            'toxicity': tox,
            'toxicity_display': TOXICITY_DISPLAY[tox],
            'n_gs_events': int(g['event'].sum()),
            'n_gs_events_30d': int(((g['event'] == 1) & (g['gs_onset_days'] <= BRIER_HORIZON_DAYS)).sum()),
            'n_llm_events': int(l['event'].sum()),
            'n_llm_events_30d': int(((l['event'] == 1) & (l['llm_onset_days'] <= BRIER_HORIZON_DAYS)).sum()),
            'ibs_30d': b.loc[tox, 'time_integrated_brier'],
            'null_ibs_30d': b.loc[tox, 'null_ibs'],
            'ibs_difference': b.loc[tox, 'ibs_difference'],
            'relative_improvement': b.loc[tox, 'relative_improvement'],
            'p_value_vs_null': b.loc[tox, 'p_value_vs_null'],
            'n_detected': int(d.loc[tox, 'n_detected']),
            'sensitivity': d.loc[tox, 'sensitivity'],
            'median_onset_diff_days': d.loc[tox, 'median_onset_diff_days'],
            'pct_within_30d': d.loc[tox, 'pct_within_30d'],
        })
cmp_df = pd.DataFrame(cmp_rows)
cmp_df.to_csv(COMPARE_CSV, index=False)
print(f'\nSaved: {os.path.basename(COMPARE_CSV)}')
print(cmp_df.to_string(index=False))

# Conservative copies already written at RESULTS_DIR root by the primary-mode saves above.
print('\nSend-set files at RESULTS_DIR root:')
for fn in sorted(os.listdir(RESULTS_DIR)):
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.isfile(p) and not fn.endswith('_all_modes.csv'):
        print(f'  {fn}')